## 1. ¿Por qué no cifrar el documento directamente con RSA?

RSA no se usa para cifrar documentos grandes directamente porque:

- Solo puede cifrar mensajes pequeños, limitados por el tamaño de la clave y el padding.
- Es mucho más lento que un cifrado simétrico como AES.
- Para archivos grandes resulta ineficiente y poco práctico.

Por eso se usa un esquema híbrido: RSA protege una clave AES aleatoria, y AES cifra el documento completo.

In [ ]:
from pathlib import Path

from generar_claves import generar_par_claves
from rsa_OAEP import cifrar_con_rsa, descifrar_con_rsa
from rsa_AES_GCM import encrypt_document, decrypt_document

## 2. Generación de claves RSA

In [3]:
private_key_pem, public_key_pem = generar_par_claves(3072)
print('Claves generadas correctamente.')
print('private_key.pem existe:', Path('private_key.pem').exists())
print('public_key.pem existe:', Path('public_key.pem').exists())

Claves generadas correctamente.
private_key.pem existe: True
public_key.pem existe: True


In [4]:
with open('public_key.pem', 'r', encoding='utf-8') as f:
    public_key_text = f.read()

print(public_key_text[:200])

-----BEGIN PUBLIC KEY-----
MIIBojANBgkqhkiG9w0BAQEFAAOCAY8AMIIBigKCAYEAwVLPDmKK96cpzIwLY2w8
y4/UKKDg24fCN9y7fw2BS4+LsClFkwmQfZTlZHw0kvlpiaM47XRsKsZoKTAI/jzo
8tQh7jkD1Or6F6WjfmwlmuLn/FTR4RiYLgSHsdg2NmT


### Respuesta

Un archivo .pem contiene datos criptográficos codificados en Base64 y delimitados por cabeceras de texto. En este caso, public_key.pem incluye:

- Una línea de inicio como -----BEGIN PUBLIC KEY-----
- El contenido de la clave codificado en Base64
- Una línea final como -----END PUBLIC KEY-----

La clave pública contiene la información necesaria para cifrar o verificar firmas, principalmente el módulo n y el exponente público e, aunque en el PEM eso aparece serializado en formato binario ASN.1 y luego codificado en Base64.

## 3. Cifrado y descifrado directo con RSA-OAEP

In [5]:
mensaje = b'El mensaje sera la clave secreta de AES'

with open('public_key.pem', 'rb') as f:
    pub = f.read()

with open('private_key.pem', 'rb') as f:
    priv = f.read()

cifrado = cifrar_con_rsa(mensaje, pub)
descifrado = descifrar_con_rsa(cifrado, priv)

print('Mensaje original:', mensaje)
print('Cifrado (hex):', cifrado.hex()[:96] + '...')
print('Descifrado:', descifrado)

Mensaje original: b'El mensaje sera la clave secreta de AES'
Cifrado (hex): 3e7eef4cd7faef429cd277bdf0cd94dad7fb92217a56742c44bf7a3f0b2f42adc4254a951bf99e144a70f3d28f172939...
Descifrado: b'El mensaje sera la clave secreta de AES'


In [6]:
c1 = cifrar_con_rsa(mensaje, pub)
c2 = cifrar_con_rsa(mensaje, pub)

print('c1 == c2:', c1 == c2)
print('Longitud c1:', len(c1))
print('Longitud c2:', len(c2))

c1 == c2: False
Longitud c1: 384
Longitud c2: 384


### Respuesta

Cifrar el mismo mensaje dos veces con RSA-OAEP produce resultados distintos porque OAEP introduce aleatoriedad en el proceso de padding antes del cifrado. Esa aleatoriedad hace que, aun con el mismo mensaje y la misma clave pública, el bloque de entrada a RSA cambie en cada ejecución.

La propiedad responsable es el padding probabilístico de OAEP. Esto mejora la seguridad porque evita que un atacante reconozca que dos cifrados corresponden al mismo mensaje.

## 4. Cifrado híbrido RSA-OAEP + AES-256-GCM

In [7]:
documento = b'Contrato de confidencialidad No. 2025-GT-001'
paquete = encrypt_document(documento, pub)
recuperado = decrypt_document(paquete, priv)

print('Tamanio del paquete cifrado:', len(paquete))
print('Documento recuperado:', recuperado)
print('Coinciden:', recuperado == documento)

Tamanio del paquete cifrado: 462
Documento recuperado: b'Contrato de confidencialidad No. 2025-GT-001'
Coinciden: True


In [8]:
import os

documento_grande = os.urandom(1024 * 1024)
paquete_grande = encrypt_document(documento_grande, pub)
recuperado_grande = decrypt_document(paquete_grande, priv)

print('Prueba de 1 MB:', recuperado_grande == documento_grande)
print('Tamanio del paquete para 1 MB:', len(paquete_grande))

Prueba de 1 MB: True
Tamanio del paquete para 1 MB: 1048994


### Explicación del paquete generado

El paquete de cifrado híbrido contiene, en este orden:

- La longitud de la clave AES cifrada con RSA
- La clave AES cifrada con RSA-OAEP
- El nonce de AES-GCM
- El tag de autenticación de AES-GCM
- El ciphertext del documento

AES-GCM aporta confidencialidad e integridad. RSA-OAEP protege la clave AES para que solo el destinatario con la clave privada pueda recuperarla.

## 5. Análisis de protocolos reales

- En TLS 1.3, RSA ya no se usa para intercambio de claves; se usa principalmente en certificados y firmas.
- En X.509, RSA aparece como algoritmo de clave pública en certificados y firmas digitales.
- En SSH, RSA puede usarse para autenticación mediante claves o firmas, aunque hoy también son comunes Ed25519 y ECDSA.

Una lección importante del laboratorio es que RSA sigue siendo útil, pero en sistemas modernos normalmente se integra con otros algoritmos en esquemas híbridos.